# Query the definitions repo with SPARQL

## Preparations

### Install

In [ ]:
%pip install rdflib

### Import

In [ ]:
import re
import requests
import rdflib
import pandas as pd

### Load files from GitHub

In [ ]:
# Repo
user = "libris"
repo = "definitions"

# Branch
branch = "feature/typenormalization"

# Get tree
api_url = f"https://api.github.com/repos/{user}/{repo}/git/trees/{branch}?recursive=1"
res = requests.get(api_url)

In [ ]:
file_list = res.json()["tree"]
print(f"Number of files in repo: {len(file_list)}")
print(f"Folders: {[f['path'] for f in file_list if f['type'] == 'tree']}")
#print(f"File: {[f['path'] for f in file_list if f['type'] == 'blob']}")


In [ ]:
folders_to_load = ["vocab", "categories", "rda"]

files = []
for fo in folders_to_load:
    folder_path = f"{fo}\/.*\.ttl"
    files.extend(f'https://raw.githubusercontent.com/{user}/{repo}/{branch}/{fp["path"]}' for fp in file_list if re.search(folder_path, fp["path"]))
    
print(*files, sep="\n")

### Use local files

### Parse content into RDF graph
Select the folders you want to load files from.

In [ ]:
# Create graph
g = rdflib.Graph()

for file in files:
  try:
    g.parse(file, format='turtle')
  except rdflib.plugins.parsers.notation3.BadSyntax as bs:
    print(bs, file)
print("\nTotal number of triples in graph:", len(g))

## Queries

### Carrier types

In [ ]:
carrierype_query = """
prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#>
prefix : <https://id.kb.se/vocab/>
prefix kbrda: <https://id.kb.se/term/rda/>
prefix ktg: <https://id.kb.se/term/ktg/>
prefix bf: <http://id.loc.gov/ontologies/bibframe/>


SELECT ?broader ?c ?type
WHERE {
?c a ?type .
?c a bf:Carrier .
OPTIONAL {?c skos:broader ?broader}
}
"""

res = g.query(carrierype_query)
carrierype_df = pd.DataFrame(res, columns=[str(var) for var in res.vars])
carrierype_df.sort_values(by="broader", inplace=True)
carrierype_df.info()
carrierype_df.head(50)

### Other instance categories

In [ ]:
instancecategory_query = """
prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#>
prefix : <https://id.kb.se/vocab/>
prefix kbrda: <https://id.kb.se/term/rda/>
prefix ktg: <https://id.kb.se/term/ktg/>
prefix bf: <http://id.loc.gov/ontologies/bibframe/>

SELECT ?broader ?c ?type
WHERE {
?c a ?type .
?type rdfs:subClassOf* :InstanceCategory .
OPTIONAL {?c skos:broader ?broader}
}
"""


res = g.query(instancecategory_query)
instancecategory_df = pd.DataFrame(res, columns=[str(var) for var in res.vars])
instancecategory_df.sort_values(by="broader", inplace=True)

instancecategory_df.info()
instancecategory_df.head(50)


### Carrier types and other instance categories combined

In [ ]:
combined_df = pd.concat([carrierype_df, instancecategory_df], axis=0)
combined_df.sort_values(by="broader", inplace=True)

combined_df.info()

with pd.option_context('display.max_rows', 100):
    display(combined_df.head(100))